# Layer 2

## Layer 2とは

**Layer 2（L2）** は、Layer 1（L1）のブロックチェーンの外でトランザクションを処理し、その結果や正しさの根拠をL1に記録することで、L1のセキュリティを借りながら処理性能を上げる仕組みの総称。

L2はL1に依存して成立する。L1に何かを書き込むことで最終的な確定を得るため、

- **スループット**（1秒あたりの処理件数）は大きく上がる
- **手数料** は1/10〜1/100程度に下がる
- ただし、L1上での最終確定までの時間はL1のブロック時間より短くはならない

という性質がある。

:::{note}
現在のL2は、L1のスケーリング手段というより独立したチェーンのように使われている。L1の資産を持ち込んで安く取引するだけでなく、L2上で新しいトークンが発行され、L2上で完結する経済圏ができている。
:::

## L2の種類

| 方式 | 概要 | 例 |
| --- | --- | --- |
| ステートチャネル | 当事者間でオフチェーンに署名済みの状態をやりとりし、開始と終了だけL1に記録する | Lightning Network（Bitcoin）, Raiden（Ethereum） |
| Plasma | 子チェーンのブロックのルートだけをL1に記録し、不正があれば退出できるようにする | （現在はほぼ使われていない） |
| Optimistic Rollup | トランザクションデータをL1に記録し、結果は正しいと仮定して受け入れ、不正があれば異議申し立てする | Arbitrum, Optimism, Base |
| ZK Rollup | トランザクションデータと、状態遷移が正しいことのゼロ知識証明をL1に記録する | zkSync Era, Starknet, Scroll, Linea |
| Validium | ZK Rollupと同様に証明を使うが、データはL1の外に置く | — |

**サイドチェーン**（Polygon PoSなど）は、独自のバリデータと合意形成を持ち、ブリッジでL1とつながっている独立したチェーン。セキュリティをL1に依存しないので、厳密にはL2に含めないことが多い。

## ステートチャネルとLightning Network

### ペイメントチャネル

AliceとBobが頻繁に送金しあう場合、毎回L1に書き込むのは無駄が多い。**ペイメントチャネル** では、

1. **開設**：2人で資金を2-of-2のマルチシグに預ける（L1のトランザクション）
2. **更新**：残高の配分を更新した「決済用トランザクション」に互いに署名して交換する（オフチェーン。何度でも、即時に行える）
3. **閉鎖**：最新の決済用トランザクションをL1に送って精算する（L1のトランザクション）

L1に記録されるのは開設と閉鎖の2回だけになる。古い（自分に有利な）状態でチャネルを閉じようとした場合、相手はそれを証明して罰として全額を没収できるようになっている。

### Lightning Network

**Lightning Network** は、ペイメントチャネルを網の目のようにつないだBitcoinのL2。直接チャネルを持っていない相手にも、チャネルを経由して送金できる。

```mermaid
flowchart LR
    A[Alice] <-->|チャネル| B[Bob] <-->|チャネル| C[Carol] <-->|チャネル| D[Dave]
```

AliceからDaveへの経由送金では、途中のノードに持ち逃げされないよう **HTLC（Hashed Timelock Contract）** を使う。Daveが秘密の値 $x$ を作ってそのハッシュ $H(x)$ をAliceに渡し、各チャネルで「期限内に $x$ を示せば支払いを受け取れる」という条件付きの支払いを連鎖させる。Daveが $x$ を明かして受け取ると、その $x$ を使って経路上の各ノードも順に受け取れるので、送金は全体として成功するか全体として失敗するかのどちらかになる（HTLCは [インターオペラビリティ](./interoperability.ipynb) のアトミックスワップでも使われる）。

ステートチャネルは、参加者が固定されている、相手がオンラインである必要がある、資金をあらかじめロックしておく必要がある、といった制約があり、汎用的なスマートコントラクトの実行には向かない。

## ロールアップ

### 考え方

ブロックチェーンがやっていることは、トランザクションを記録し、それを実行した結果のステートを管理することである。**ロールアップ（rollup）** は、これをL1上のスマートコントラクトとオフチェーンの計算を組み合わせて、L1とは別のトランザクションとステートの体系として実現する（その意味ではロールアップはDAppsの一種とも言える）。

素朴に考えると、

- **案1**：多数のL2トランザクションを1つのL1トランザクションに詰め込み、L1のコントラクトで全部実行してL2のステートを更新する → 計算がL1で行われるので安くならない
- **案2**：L2トランザクションのデータだけをL1に記録し、実行とステートの計算はオフチェーンで行い、ステートのルートだけをL1に記録する → 安くなるが、記録されたステートルートが本当に正しいのか分からない

ロールアップは案2を採用し、「ステートルートの正しさ」を次のどちらかの方法で保証する。

| 方式 | 正しさの保証 |
| --- | --- |
| Optimistic Rollup | 結果は正しいと **楽観的に** 仮定して受け入れ、一定期間内に誰かが不正を証明すれば取り消す（**フラウドプルーフ**, fraud proof） |
| ZK Rollup | 状態遷移が正しいことを示す **ゼロ知識証明**（**バリディティプルーフ**, validity proof）を添付し、L1のコントラクトで検証する |

どちらの方式でも、**L2のトランザクションデータ自体はL1に記録される** のが重要な点。L2の運営者がいなくなっても、誰でもL1上のデータからL2のステートを再構築でき、資金を引き出せる。

```mermaid
flowchart TB
    U["ユーザー"] -->|"L2 Tx"| Seq["シーケンサー<br/>（順序付け・実行）"]
    Seq -->|"Txデータを圧縮してバッチで投稿"| L1Data["L1: データ（calldata / blob）"]
    Seq -->|"ステートルート（+証明）"| L1Contract["L1: ロールアップのコントラクト"]
    V["検証者"] -->|"データを読んで再実行"| L1Data
    V -->|"不正なら異議申し立て<br/>（Optimistic）"| L1Contract
```

## Optimistic Rollup（Arbitrum Oneの例）

Arbitrum Oneは、Offchain Labsが開発し2021年に稼働したOptimistic Rollup。2023年からはArbitrum DAOがガバナンスを担っている。

### 登場人物

| 役割 | 内容 |
| --- | --- |
| シーケンサー | ユーザーからL2トランザクションを受け取って順序を決め、実行結果を即座に返す。トランザクションをまとめて圧縮し、バッチとしてL1のInboxコントラクトに投稿する。現在は1者（Offchain Labs）が運用 |
| バリデータ | L1に投稿されたトランザクション列を独立に実行してL2のステートを計算し、その結果を **Assertion** としてL1のRollupコントラクトに提案する。他者の不正なAssertionに異議を申し立てる |

ユーザーから受け取るトランザクションには署名が付いているので、シーケンサーが内容を改ざんすることはできない。シーケンサーにできるのは順序の決定と、トランザクションを取り込まない（検閲する）ことである。検閲に対しては、ユーザーがL1のInboxに直接トランザクションを投稿する経路（delayed inbox）が用意されている。

### 処理の流れ

1. ユーザーがL1と同じ形式のL2トランザクションを作り、シーケンサーに送る
2. シーケンサーが実行して結果を返す（**ソフトコンファメーション**。シーケンサーを信頼する前提の速報値）
3. シーケンサーがトランザクションのバッチをL1に投稿する（L1上での順序の確定）
4. バリデータがバッチを実行し、Assertion（複数のL2ブロックにわたる実行結果の主張）をL1に提案する
5. **チャレンジ期間（7日間）** 中に異議がなければ、Assertionが確定する

### フラウドプルーフ

不正なAssertionを見つけたバリデータは異議を申し立てられる。1つのL2トランザクションでも内部では膨大な数の命令が実行されるため、全部をL1で再実行するのは高すぎる。そこで、

1. 提案者と異議申立者が、実行の途中経過のハッシュを示し合いながら、主張が食い違う範囲を **二分探索** で絞り込む
2. 最終的に1つの命令（ArbitrumではWAVMの1命令）まで絞り込む
3. その1ステップだけをL1のコントラクトで実行して、どちらが正しいかを判定する（**ワンステッププルーフ**）

$n$ ステップの実行でも、やりとりは $O(\log n)$ 回で済む。

In [1]:
import hashlib


def step(state: int) -> int:
    """1ステップの状態遷移（例として適当な決定論的関数）"""
    return (state * 6364136223846793005 + 1442695040888963407) % 2**64


def trace(initial: int, n: int, corrupt_at: int | None = None) -> list[int]:
    """各ステップ後の状態の列。corrupt_atステップ目で不正な遷移を混ぜる"""
    states = [initial]
    for i in range(n):
        s = step(states[-1])
        if i == corrupt_at:
            s ^= 1  # 1ビットだけ改ざん
        states.append(s)
    return states


def commit(s: int) -> str:
    return hashlib.sha256(s.to_bytes(8, "big")).hexdigest()[:12]


N_STEPS = 1_000_000
honest = trace(42, N_STEPS)
dishonest = trace(42, N_STEPS, corrupt_at=777_777)

# 両者とも最終状態が食い違っている。二分探索で最初に食い違うステップを探す
lo, hi = 0, N_STEPS  # lo では一致、hi では不一致
rounds = 0
while hi - lo > 1:
    mid = (lo + hi) // 2
    rounds += 1
    if commit(honest[mid]) == commit(dishonest[mid]):
        lo = mid
    else:
        hi = mid

print(f"{N_STEPS:,} ステップの実行に対して {rounds} 回のやりとりで特定: ステップ {lo} → {hi}")
# L1では、合意している状態から1ステップだけ実行して判定する
print("L1での1ステップ検証: 正しい結果 =", step(dishonest[lo]) == honest[hi], "/ 不正な結果 =", step(dishonest[lo]) == dishonest[hi])

1,000,000 ステップの実行に対して 20 回のやりとりで特定: ステップ 777777 → 777778
L1での1ステップ検証: 正しい結果 = True / 不正な結果 = False


### 引き出しとチャレンジ期間

L2からL1へ資産を引き出す場合、L2で引き出しのトランザクションを実行した後、それを含むAssertionがチャレンジ期間を経て確定するのを待ち、L1のOutboxコントラクトでマークルプルーフを示して受け取る。そのため、**Optimistic Rollupからの引き出しには約7日かかる**。

実際には、流動性提供者が手数料と引き換えにL1側で即座に立て替えてくれるサービス（高速ブリッジ）がよく使われる。

### 手数料

L2の手数料は次の2つからなる。

$$
\text{L2の手数料} = \text{L2での実行費用} + \text{L1へのデータ投稿費用}
$$

ユーザーはL2上のETHで支払い、シーケンサーはL1へのデータ投稿のためにL1のETHでgasを払う。EIP-4844以前は、手数料の大部分がL1へのデータ投稿費用だった。

## ZK Rollup

**ZK Rollup** は、オペレーター（プルーバー）がL2トランザクションをまとめて実行し、「前のステートルートから、これらのトランザクションを正しく実行すると新しいステートルートになる」ことのゼロ知識証明（SNARK/STARK）を作ってL1に提出する。L1のコントラクトは証明を検証するだけでよく、検証のコストはトランザクション数にほとんど依存しない。

### 送金だけのZK Rollupの設計例

ETHの送金だけができる単純なZK Rollupを考える。

- **ステート**：公開鍵をキー、残高を値とするマークルツリー
- **ユーザー**：送金元・送金先・金額を含むL2トランザクションに、ZKで検証しやすい署名方式（EdDSAなど）で署名し、オペレーターに送る
- **オペレーター**：例えば1,000件のL2トランザクションをまとめ、次の処理を行うZK回路の証明を作る
  1. 実行前のステートのマークルプルーフから、前のステートルートと一致することを確認
  2. 各トランザクションの署名を検証し、残高が足りることを確認
  3. 送金を実行してステートを更新
  4. 新しいステートルートを計算
- **L1のコントラクト**：トランザクションデータ（署名を除く）、新旧のステートルート、証明を受け取り、証明を検証する

署名はサイズが大きいがL1には記録しなくてよい。署名が正しいことは証明に含まれているからである。

### Optimistic RollupとZK Rollupの比較

| 項目 | Optimistic Rollup | ZK Rollup |
| --- | --- | --- |
| 正しさの保証 | フラウドプルーフ（事後） | バリディティプルーフ（事前） |
| 確定 | チャレンジ期間（約7日）後 | 証明がL1で検証された時点（数分〜数時間） |
| L1への引き出し | 約7日 | 証明の検証後すぐ |
| 安全性の前提 | 少なくとも1人の正直な検証者がいること | 暗号の安全性（とZK回路にバグがないこと） |
| オフチェーンの計算コスト | 小さい | 証明の生成が重い |
| EVM互換性 | 高い（既存のクライアントをほぼそのまま使える） | zkEVMの実装が難しかったが、急速に改善している |

**zkEVM** は、EVMの実行そのものをZKで証明するもの。EVMとの互換性の度合いによって、Ethereumと完全に同等なもの（Type 1）から、言語レベルの互換性にとどまるもの（Type 4）まで分類される（Vitalikによる分類）。

## データ可用性

### データ可用性問題

ロールアップの安全性は、「L2のトランザクションデータを誰でも入手できる」ことに依存している。データが公開されていなければ、

- Optimistic Rollupでは不正を見つけて異議を申し立てられない
- ZK Rollupでも、ステートが正しいことは分かっても中身が分からず、運営者がいなくなると資金を引き出せない

これを **データ可用性（Data Availability, DA）問題** と呼ぶ。

当初、ロールアップはトランザクションデータをL1トランザクションの **calldata** として投稿していた。calldataは全ノードが永久に保持するデータなので、1バイトあたり16 gasと高価で、L2の手数料の大部分を占めていた。

### EIP-4844（Proto-Danksharding）

2024年3月のDencunアップグレードで導入された **EIP-4844（Proto-Danksharding）** は、ロールアップ向けの安価なデータ領域 **blob** を追加した。

| 項目 | calldata | blob |
| --- | --- | --- |
| 保持期間 | 永久 | 約18日（4,096エポック）で削除される |
| EVMからのアクセス | 可能 | 不可（コミットメントのハッシュだけ参照できる） |
| 料金 | 通常のgas市場 | blob専用の手数料市場（blob gas） |
| 保持する場所 | 実行レイヤー | コンセンサスレイヤー（BlobSidecar） |

- 1つのblobは約128KB
- ブロックあたりのblob数は当初ターゲット3・最大6で、Pectra（2025年）でターゲット6・最大9に増えた
- トランザクションにはblobの **KZGコミットメント** のハッシュ（versioned hash）が含まれ、blobの中身がそのとき記録されたものと一致することを検証できる

blobは18日で消えるが、Optimistic Rollupのチャレンジ期間（7日）より長いので、その間に誰でもデータを取得・検証できれば十分という考え方である。

導入後、L2の手数料は大きく下がった。blob専用の手数料市場が混雑していない間は、blobの料金は最低価格（1 wei / blob gas）まで下がり、ほぼ無料になっていた（Fusakaアップグレードで、実行レイヤーのbase feeに連動した下限が設けられた。EIP-7918）。

In [2]:
# calldata と blob にL2のデータ 100KB を投稿する場合のコスト比較（概算）
DATA_BYTES = 100 * 1024

calldata_gas = DATA_BYTES * 16  # 非ゼロバイトは16 gas/byte
base_fee_gwei = 1.0  # 実行レイヤーのbase fee（仮定）
calldata_eth = calldata_gas * base_fee_gwei * 1e-9

BLOB_SIZE = 131_072  # 1 blob = 4096 * 32 bytes
n_blobs = -(-DATA_BYTES // BLOB_SIZE)  # 切り上げ
blob_gas = n_blobs * BLOB_SIZE  # 1 blob gas / byte
for blob_base_fee_gwei in [1e-9, 1e-3, 1.0]:
    blob_eth = blob_gas * blob_base_fee_gwei * 1e-9
    print(f"blob base fee {blob_base_fee_gwei:g} gwei: blob {blob_eth:.2e} ETH")
print(f"calldata (base fee {base_fee_gwei} gwei): {calldata_eth:.2e} ETH, {calldata_gas:,} gas")

blob base fee 1e-09 gwei: blob 1.31e-13 ETH
blob base fee 0.001 gwei: blob 1.31e-07 ETH
blob base fee 1 gwei: blob 1.31e-04 ETH
calldata (base fee 1.0 gwei): 1.64e-03 ETH, 1,638,400 gas


### Dankshardingとデータ可用性サンプリング

blobの数を大きく増やすと、全ノードがすべてのblobをダウンロードするのは負担が大きい。そこで、

- blobを **イレイジャーコーディング**（リード・ソロモン符号）で冗長化して、一部が欠けても元に戻せるようにする
- 各ノードはblobのランダムな一部だけをダウンロードし、それが入手できればblob全体が公開されていると高い確率で判断する（**データ可用性サンプリング, DAS**）

という仕組みで、ノードの負担を増やさずにデータ容量を増やす。2025年12月のFusakaアップグレードで、その第一段階である **PeerDAS**（EIP-7594）が導入された。最終形は **Danksharding** と呼ばれる。

## L2のリスクと分散化

多くのL2は、まだ完全にトラストレスとは言えない。

| リスク | 内容 |
| --- | --- |
| シーケンサーの集中 | 1者がシーケンサーを運用しているため、検閲や停止、順序の操作（MEV）が可能 |
| アップグレード権限 | L1上のコントラクトを運営チームのマルチシグが即座にアップグレードできると、資金を動かせてしまう |
| 証明システム | フラウドプルーフが許可された者しか出せない、または未実装のL2もある |

[L2BEAT](https://l2beat.com/) は、L2の分散化の度合いを **Stage 0〜2** で評価している。

| ステージ | 状態 |
| --- | --- |
| Stage 0 | 運営者による管理が中心（補助輪付き）。データはL1にあり、ステートを再構築できる |
| Stage 1 | 証明システムが稼働しており、運営者の介入は、一定以上の参加者からなるセキュリティ委員会に限られる |
| Stage 2 | 証明システムが完全に許可不要で、ユーザーは不正なアップグレードの前に退出する猶予がある |

シーケンサーの分散化として、複数のシーケンサーで順番に担当する方式や、L1のバリデータに順序付けを任せる **based rollup** などが研究されている。

## 参考文献

- [Scaling | ethereum.org](https://ethereum.org/developers/docs/scaling/)
- Vitalik Buterin. [An Incomplete Guide to Rollups](https://vitalik.eth.limo/general/2021/01/05/rollup.html)
- Vitalik Buterin. [The different types of ZK-EVMs](https://vitalik.eth.limo/general/2022/08/04/zkevm.html)
- [Arbitrum Nitro Whitepaper](https://github.com/OffchainLabs/nitro/blob/master/docs/Nitro-whitepaper.pdf)
- [EIP-4844: Shard Blob Transactions](https://eips.ethereum.org/EIPS/eip-4844)
- [L2BEAT](https://l2beat.com/)
- Poon, J. & Dryja, T. (2016). [The Bitcoin Lightning Network](https://lightning.network/lightning-network-paper.pdf)